In [0]:
# ============================================================
# NOTEBOOK 03 — SILVER: EDA con SparkSQL
# Vermont School - Sistema de Alerta Temprana 2025-26
# ============================================================

TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

# Leer desde Trusted y registrar como tabla temporal
df_spark = spark.read.parquet(f"{TRUSTED}/estudiantes_2025_26")
df_spark.createOrReplaceTempView("estudiantes")

print(f"✓ Tabla 'estudiantes' registrada")
print(f"  Filas: {df_spark.count()}")
print(f"  Columnas: {len(df_spark.columns)}")

In [0]:
# CELDA 2 — Distribución general por grado y nivel de riesgo
resultado_1 = spark.sql("""
    SELECT 
        grado,
        nivel_riesgo,
        COUNT(*) as n_estudiantes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY grado), 1) as pct_en_grado
    FROM estudiantes
    GROUP BY grado, nivel_riesgo
    ORDER BY grado, nivel_riesgo
""")

print("=== DISTRIBUCIÓN DE RIESGO POR GRADO ===")
resultado_1.show(20, truncate=False)

# Guardar en Silver
resultado_1.write.mode("overwrite").parquet(f"{SILVER}/eda_riesgo_por_grado")
print("✓ Guardado en Silver: eda_riesgo_por_grado")

In [0]:
# CELDA 3 — Estadísticas de notas y ausencias por grado
resultado_2 = spark.sql("""
    SELECT
        grado,
        COUNT(*) as n_estudiantes,
        ROUND(AVG(promedio_acumulado), 2) as promedio_notas,
        ROUND(MIN(promedio_acumulado), 2) as min_notas,
        ROUND(MAX(promedio_acumulado), 2) as max_notas,
        ROUND(AVG(total_ausencias), 1)   as promedio_ausencias,
        ROUND(AVG(n_f1), 1)              as promedio_f1,
        MAX(n_f1)                        as max_f1,
        SUM(CASE WHEN n_f2 > 0 THEN 1 ELSE 0 END) as estudiantes_con_f2
    FROM estudiantes
    GROUP BY grado
    ORDER BY grado
""")

print("=== ESTADÍSTICAS POR GRADO ===")
resultado_2.show(truncate=False)

resultado_2.write.mode("overwrite").parquet(f"{SILVER}/eda_estadisticas_por_grado")
print("✓ Guardado en Silver: eda_estadisticas_por_grado")

In [0]:
# CELDA 4 — Análisis de ausencias vs nivel de riesgo
resultado_3 = spark.sql("""
    SELECT
        nivel_riesgo,
        COUNT(*) as n_estudiantes,
        ROUND(AVG(total_ausencias), 1)        as promedio_ausencias,
        ROUND(AVG(ausencia_clase), 1)         as prom_ausencia_clase,
        ROUND(AVG(retardo), 1)                as prom_retardos,
        ROUND(AVG(n_f1), 1)                   as promedio_f1,
        ROUND(AVG(n_f2), 2)                   as promedio_f2,
        ROUND(AVG(promedio_acumulado), 2)     as promedio_notas
    FROM estudiantes
    GROUP BY nivel_riesgo
    ORDER BY 
        CASE nivel_riesgo
            WHEN 'riesgo_critico'      THEN 1
            WHEN 'riesgo_recuperacion' THEN 2
            WHEN 'sin_riesgo'          THEN 3
        END
""")

print("=== AUSENCIAS Y SEGUIMIENTOS POR NIVEL DE RIESGO ===")
resultado_3.show(truncate=False)

resultado_3.write.mode("overwrite").parquet(f"{SILVER}/eda_ausencias_vs_riesgo")
print("✓ Guardado en Silver: eda_ausencias_vs_riesgo")

In [0]:
# CELDA 5 — Top asignaturas con más estudiantes bajo 4.0
resultado_4 = spark.sql("""
    SELECT asignatura, 
           SUM(CASE WHEN nota < 4.0 THEN 1 ELSE 0 END) as n_bajo_4,
           COUNT(*) as n_total,
           ROUND(AVG(nota), 2) as promedio,
           ROUND(MIN(nota), 2) as minimo
    FROM (
        SELECT id_estudiante,
               stack(11,
                'Integrated Science',      `nota_acum_Integrated Science`,
                'Individuals and societies',`nota_acum_Individuals and societies`,
                'Educación Física',         `nota_acum_Educación Física`,
                'English',                  `nota_acum_English`,
                'Lengua Castellana',        `nota_acum_Lengua Castellana`,
                'Mandarín',                 `nota_acum_Mandarín`,
                'Mathematics',             `nota_acum_Mathematics`,
                'Financial Maths',         `nota_acum_Financial Maths`,
                'ICT/STEM',                `nota_acum_ICT/STEM`,
                'Life Science',            `nota_acum_Life Science`,
                'Physical Science',        `nota_acum_Physical Science`
               ) as (asignatura, nota)
        FROM estudiantes
    )
    WHERE nota IS NOT NULL
    GROUP BY asignatura
    ORDER BY n_bajo_4 DESC
""")

print("=== ASIGNATURAS CON MÁS ESTUDIANTES BAJO 4.0 ===")
resultado_4.show(15, truncate=False)

resultado_4.write.mode("overwrite").parquet(f"{SILVER}/eda_riesgo_por_asignatura")
print("✓ Guardado en Silver: eda_riesgo_por_asignatura")

In [0]:
# CELDA 6 — Correlaciones y resumen final EDA
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

df_pd = spark.read.parquet(f"{TRUSTED}/estudiantes_2025_26").toPandas()

# Correlaciones entre variables numéricas clave
vars_clave = [
    'promedio_acumulado', 'nota_min_acumulada',
    'total_ausencias', 'ausencia_clase',
    'retardo', 'n_f1', 'n_f2', 'n_asig_bajo_4'
]
corr = df_pd[vars_clave].corr().round(2)

print("=== CORRELACIONES ENTRE VARIABLES CLAVE ===")
print(corr.to_string())

# Gráfico 1: Distribución de riesgo por grado
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Vermont School 2025-26 — EDA', fontsize=14, fontweight='bold')

orden_riesgo = ['sin_riesgo', 'riesgo_recuperacion', 'riesgo_critico']
colores = ['#2ecc71', '#f39c12', '#e74c3c']

for i, grado in enumerate(['7', '8', '9']):
    df_g = df_pd[df_pd['grado'] == grado]
    conteos = [len(df_g[df_g['nivel_riesgo'] == r]) for r in orden_riesgo]
    etiquetas = ['Sin riesgo', 'Recuperación', 'Crítico']
    axes[i].bar(etiquetas, conteos, color=colores)
    axes[i].set_title(f'Grado {grado}°')
    axes[i].set_ylabel('N° estudiantes')
    for j, v in enumerate(conteos):
        axes[i].text(j, v + 0.2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/eda_riesgo_grados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado")

# Gráfico 2: Ausencias promedio por nivel de riesgo
fig2, ax = plt.subplots(figsize=(8, 5))
promedios = df_pd.groupby('nivel_riesgo')['total_ausencias'].mean()
promedios = promedios.reindex(orden_riesgo)
etiquetas = ['Sin riesgo', 'Recuperación', 'Crítico']
bars = ax.bar(etiquetas, promedios.values, color=colores)
ax.set_title('Promedio de ausencias por nivel de riesgo', fontweight='bold')
ax.set_ylabel('Total ausencias')
for bar, val in zip(bars, promedios.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/eda_ausencias_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico 2 guardado")

# Guardar resumen EDA en Silver
resumen_eda = spark.createDataFrame(df_pd[vars_clave + ['nivel_riesgo', 'grado']])
resumen_eda.write.mode("overwrite").parquet(f"{SILVER}/eda_resumen_correlaciones")
print(f"\n✓ EDA completo — todos los resultados guardados en Silver")